# Assignment 3 — Module 3: Evolutionary and Swarm Optimization


## Practical 4: Crossover Operations for Generating Offspring

### Aim
To implement the crossover operation for generating new offspring from selected individuals using Python.

### Theory
**Crossover** combines genetic material from two parents to create offspring that inherit parts of each parent. Its usual probability is $p_c \approx 0.6$–$0.9$.

- **Single-point crossover (binary):** cut both parents at one random position and swap the tails.
- **Two-point crossover (binary):** swap the segment between two random cut points.
- **Uniform crossover (binary):** each gene is taken from either parent with probability $0.5$.
- **Arithmetic crossover (real-valued):** $c_1 = \alpha p_1 + (1-\alpha)p_2$, $c_2 = (1-\alpha)p_1 + \alpha p_2$.

### Binary crossover operators

In [1]:
import numpy as np

rng = np.random.default_rng(7)

def single_point_crossover(p1, p2):
    point = rng.integers(1, len(p1))            # cut between 1 and L-1
    c1 = np.concatenate([p1[:point], p2[point:]])
    c2 = np.concatenate([p2[:point], p1[point:]])
    return c1, c2

def two_point_crossover(p1, p2):
    a, b = sorted(rng.choice(np.arange(1, len(p1)), size=2, replace=False))
    c1, c2 = p1.copy(), p2.copy()
    c1[a:b], c2[a:b] = p2[a:b], p1[a:b]        # swap the middle segment
    return c1, c2

def uniform_crossover(p1, p2):
    mask = rng.integers(0, 2, size=len(p1)).astype(bool)
    c1 = np.where(mask, p1, p2)
    c2 = np.where(mask, p2, p1)
    return c1, c2

### Real-valued arithmetic crossover

In [2]:
def arithmetic_crossover(p1, p2, alpha=None):
    if alpha is None:
        alpha = rng.random()                    # random blending weight in [0, 1]
    c1 = alpha * p1 + (1 - alpha) * p2
    c2 = (1 - alpha) * p1 + alpha * p2
    return c1, c2

### Demonstrating crossover on parents

In [3]:
p1 = np.array([1, 1, 1, 1, 1, 1, 1, 1])
p2 = np.array([0, 0, 0, 0, 0, 0, 0, 0])
print("Parent 1 :", p1, "\nParent 2 :", p2)

for name, op in [('Single-point', single_point_crossover),
                 ('Two-point', two_point_crossover),
                 ('Uniform', uniform_crossover)]:
    c1, c2 = op(p1, p2)
    print(f"\n{name} crossover\n  child 1: {c1}\n  child 2: {c2}")

r1 = np.array([2.0, -1.0, 3.0])
r2 = np.array([-2.0, 4.0, 0.0])
c1, c2 = arithmetic_crossover(r1, r2, alpha=0.3)
print("\nArithmetic crossover (alpha=0.3)")
print("  parents :", r1, r2)
print("  children:", np.round(c1, 3), np.round(c2, 3))

Parent 1 : [1 1 1 1 1 1 1 1] 
Parent 2 : [0 0 0 0 0 0 0 0]

Single-point crossover
  child 1: [1 1 1 1 1 1 1 0]
  child 2: [0 0 0 0 0 0 0 1]

Two-point crossover
  child 1: [1 1 1 1 0 1 1 1]
  child 2: [0 0 0 0 1 0 0 0]

Uniform crossover
  child 1: [1 1 1 0 0 0 0 1]
  child 2: [0 0 0 1 1 1 1 0]

Arithmetic crossover (alpha=0.3)
  parents : [ 2. -1.  3.] [-2.  4.  0.]
  children: [-0.8  2.5  0.9] [0.8 0.5 2.1]


### Measuring how often crossover changes parents

In [4]:
# Crossover should create new variation; count how many children differ from both parents
trials = 1000
changed = 0
for _ in range(trials):
    a = rng.integers(0, 2, 12); b = rng.integers(0, 2, 12)
    c1, _ = single_point_crossover(a, b)
    if not (np.array_equal(c1, a) or np.array_equal(c1, b)):
        changed += 1
print(f"Single-point crossover produced a new child in {changed/trials*100:.1f}% of {trials} trials")

Single-point crossover produced a new child in 79.7% of 1000 trials


### Observation
- In the demonstration, one-point and two-point crossover produce children that are mixtures of the two parents, while uniform crossover mixes genes position-by-position.
- Arithmetic crossover on real values produces children lying between the parents along the line joining them.
- Most random crossovers create a genuinely new child, which confirms that crossover adds variation to the population.

### Conclusion
Implemented single-point, two-point, uniform and arithmetic crossover, and verified that offspring combine genetic material from both parents.